# Isolation Forest - multi-run experiments

In [ ]:
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

sys.path.append("../src")
from data import make_optuna_subsample, make_final_subsample
from optuna_utils import run_study
from metrics import find_best_f1_threshold, minmax_scale_scores, evaluate_scores, print_metrics
from results import build_experiment_record, save_record_json, get_memory_mb

In [ ]:
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ["CICIDS", "UNSW_NB15"]
SEED = 29
N_TRIALS = 200

DATASET_VERSION = "v1"
PREPROCESSING_VERSION = "v1"
SPLIT_METHOD = "stratified_train_val_test_fixed_seed"
MODEL_TYPE = "IsolationForest"
FUSION_STRATEGY = "none"

RUN_CONFIGS = [
    dict(run_index=1, n_train_opt=7000, n_val_opt=3000, n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run1_small_opt_sample"),
    dict(run_index=2, n_train_opt=21000, n_val_opt=9000, n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run2_medium_opt_sample"),
    dict(run_index=3, n_train_opt=35000, n_val_opt=15000, n_train_final=154000, n_val_final=66000, n_test_final=100000, notes="run3_large_opt_sample"),
]

In [ ]:
def make_objective(train_x, val_x, val_y):
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 600, step=100),
            "max_samples": trial.suggest_float("max_samples", 0.1, 1.0),
            "max_features": trial.suggest_float("max_features", 0.5, 1.0),
            "contamination": trial.suggest_float("contamination", 0.01, 0.5),
            "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
        }
        clf = IsolationForest(**params, random_state=SEED, n_jobs=-1)
        clf.fit(train_x)
        val_scores = -clf.score_samples(val_x)
        return roc_auc_score(val_y, val_scores)
    return objective


def fit_and_score_if(params, train_x, val_x, test_x):
    clf = IsolationForest(**params, random_state=SEED, n_jobs=-1)
    mem_before = get_memory_mb()
    start_train = time.time()
    clf.fit(train_x)
    runtime_train = time.time() - start_train
    mem_after_train = get_memory_mb()

    val_scores = minmax_scale_scores(-clf.score_samples(val_x))
    start_inference = time.time()
    test_scores = minmax_scale_scores(-clf.score_samples(test_x))
    runtime_inference = time.time() - start_inference
    mem_after_inference = get_memory_mb()
    memory_peak = max(mem_before, mem_after_train, mem_after_inference)
    return clf, val_scores, test_scores, runtime_train, runtime_inference, memory_peak


def save_if_model(clf, dataset, run_index):
    model_dir = MODELS_DIR / MODEL_TYPE
    model_dir.mkdir(parents=True, exist_ok=True)
    model_path = model_dir / f"{dataset}_run{run_index}.joblib"
    joblib.dump(clf, model_path)
    return model_path

In [ ]:
def run_experiment(dataset, run_cfg):
    run_index = run_cfg["run_index"]
    study_name = f"IF_{dataset}_run{run_index}"

    train_x, train_y, val_x, val_y = make_optuna_subsample(dataset, SEED, run_cfg["n_train_opt"], run_cfg["n_val_opt"])
    study = run_study(make_objective(train_x, val_x, val_y), study_name, SEED, N_TRIALS, results_dir=RESULTS_DIR)

    train_x, train_y, val_x, val_y, test_x, test_y = make_final_subsample(
        dataset, SEED, run_cfg["n_train_final"], run_cfg["n_val_final"], run_cfg["n_test_final"]
    )

    clf, scores_val, scores_test, runtime_train, runtime_inference, memory_peak = fit_and_score_if(study.best_params, train_x, val_x, test_x)
    best_threshold, best_f1_val, best_prec_val, best_rec_val = find_best_f1_threshold(val_y, scores_val)
    metrics = evaluate_scores(test_y, scores_test, threshold=best_threshold)
    print_metrics(f"IsolationForest final - {dataset} run{run_index}", metrics)

    model_path = save_if_model(clf, dataset, run_index)
    record = build_experiment_record(
        dataset_name=dataset, dataset_version=DATASET_VERSION, split_method=SPLIT_METHOD, seed=SEED,
        preprocessing_version=PREPROCESSING_VERSION, model_type=MODEL_TYPE, fusion_strategy=FUSION_STRATEGY,
        hyperparameters=study.best_params, threshold=best_threshold, scores_test=scores_test, test_y=test_y,
        runtime_train=runtime_train, runtime_inference=runtime_inference, memory_peak=memory_peak,
        notes=run_cfg["notes"],
        threshold_info={"selection_method": "validation_f1_max", "f1": best_f1_val, "precision": best_prec_val, "recall": best_rec_val},
        model_path=model_path,
    )
    save_record_json(record, RESULTS_DIR, run_index, MODEL_TYPE, dataset)
    return record

In [ ]:
all_records = []

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[0])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[0])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[1])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[1])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[0], RUN_CONFIGS[2])
all_records.append(rec)
pd.DataFrame(all_records)

In [ ]:
rec = run_experiment(DATASETS[1], RUN_CONFIGS[2])
all_records.append(rec)
pd.DataFrame(all_records)